In [8]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import json, glob, os
from PIL import Image


# Path to one of your spatial binning levels
base_dir = "./Prostate_spatial_data"

# finding unique samplle prefixes
samples = sorted(set(
    "_".join(os.path.basename(f).split("_")[:3])
    for f in glob.glob(os.path.join(base_dir, "*_matrix.mtx"))
))

print(samples)

adatas = {}

for sample in samples:
    print(f"Loading {sample} ...")
    prefix = os.path.join(base_dir, sample)
    
    # Read 10x count matrix
    ad = sc.read_mtx(prefix + "_matrix.mtx").T
    ad.var_names = pd.read_csv(prefix + "_features.tsv", sep="\t", header=None)[1]
    ad.obs_names = pd.read_csv(prefix + "_barcodes.tsv", sep="\t", header=None)[0]
    
    #  Load spatial coordinates 
    pos = pd.read_csv(prefix + "_tissue_positions_list.csv", header=None)
    pos.columns = ["barcode", "in_tissue", "array_row", "array_col", "pxl_row_in_fullres", "pxl_col_in_fullres"]
    pos.index = pos["barcode"]

    #  keep only barcodes that exist in ad.obs_names
    pos = pos.loc[pos["barcode"].isin(ad.obs_names), :]

    # ensure ordering matches AnnData obs
    pos = pos.reindex(ad.obs_names)

    # Join metadata and assign spatial coordinates
    ad.obs = ad.obs.join(pos, how="left")
    ad.obsm["spatial"] = pos[["pxl_row_in_fullres", "pxl_col_in_fullres"]].to_numpy(dtype=float)

    # --- Load scalefactors ---
    with open(prefix + "_scalefactors_json.json") as f:
        ad.uns["spatial"] = {sample: {"scalefactors": json.load(f)}}
    
    # Load images 
    img = Image.open(prefix + "_tissue_lowres_image.png")
    ad.uns["spatial"][sample]["images"] = {"lowres": np.array(img)}
    
    ad.uns["spatial"][sample]["metadata"] = {"source": sample}
    
    adatas[sample] = ad

print(f"\nLoaded {len(adatas)} spatial samples.")


['GSM8557976_BPH_1', 'GSM8557977_BPH_2', 'GSM8557978_BPH_3', 'GSM8557979_BPH_4', 'GSM8557980_TRNA_1', 'GSM8557981_TRNA_2', 'GSM8557982_TRNA_3', 'GSM8557983_TRNA_4', 'GSM8557984_TRNA_5', 'GSM8557985_TRNA_6', 'GSM8557986_TRNA_7', 'GSM8557987_TRNA_8', 'GSM8557988_TRNA_9', 'GSM8557989_TRNA_10', 'GSM8557990_TRNA_11', 'GSM8557991_TRNA_12', 'GSM8557992_TRNA_13', 'GSM8557993_TRNA_14', 'GSM8557994_TRNA_15', 'GSM8557995_TRNA_16', 'GSM8557996_TRNA_17', 'GSM8557997_NEADT_1', 'GSM8557998_NEADT_2', 'GSM8557999_NEADT_3', 'GSM8558000_NEADT_4', 'GSM8558001_NEADT_5', 'GSM8558002_NEADT_6', 'GSM8558003_NEADT_7', 'GSM8558004_NEADT_8', 'GSM8558005_NEADT_9', 'GSM8558006_NEADT_10', 'GSM8558007_NEADT_11', 'GSM8558008_NEADT_12', 'GSM8558009_NEADT_13', 'GSM8558010_NEADT_14', 'GSM8558011_NEADT_15', 'GSM8558012_NEADT_16', 'GSM8558013_NEADT_17', 'GSM8558014_NEADT_18', 'GSM8558015_NEADT_19', 'GSM8558016_NEADT_20', 'GSM8558017_NEADT_21', 'GSM8558018_NEADT_22', 'GSM8558019_CRPC_1', 'GSM8558020_CRPC_2', 'GSM8558021_CRP

In [ ]:
import matplotlib.pyplot as plt

gene = "AR"

for sample in samples:
    ad = adatas[sample]

    # === 1. Extract spatial info ===
    spatial_info = ad.uns["spatial"][sample]
    img = spatial_info["images"]["lowres"]

    # === 2. Get scalefactor and coordinates ===
    sf_dict = spatial_info.get("scalefactors", {})
    scale_lowres = sf_dict.get("tissue_lowres_scalef", 1.0)
    coords = ad.obsm["spatial"] * scale_lowres  # scale coords



    if gene in ad.var_names:
        # Extract gene expression s
        x = ad[:, gene].X
        values = x.toarray().flatten() if hasattr(x, "toarray") else x.flatten()

        # Plot overlay
        plt.figure(figsize=(6,6))
        plt.imshow(img)
        sc = plt.scatter(coords[:,0], coords[:,1],
                         c=values,
                         cmap="viridis",
                         s=8,
                         alpha=0.8)
        plt.colorbar(sc, label=f"{gene} expression")
        plt.title(f"{sample} — {gene}")
        plt.axis("off")
        plt.show()
    else:
        print(f" Gene {gene} not found in {sample}")



In [ ]:
import matplotlib.pyplot as plt

gene = "AR"

for sample in samples:
    ad = adatas[sample]

    # === 1. Extract spatial info ===
    spatial_info = ad.uns["spatial"][sample]
    img = spatial_info["images"]["lowres"]

    # === 2. Get scalefactor and coordinates ===
    sf_dict = spatial_info.get("scalefactors", {})
    scale_lowres = sf_dict.get("tissue_lowres_scalef", 1.0)
    coords = ad.obsm["spatial"] * scale_lowres  # scale coords


    # Swap axes (rotate 90 degrees clockwise)
    coords_rot = coords.copy()

    # Invert y-axis first
    coords_rot[:, 1] = img.shape[0] - coords[:, 1]

    coords_rot = coords_rot[:, [1, 0]]  # swap x and y
    
    coords_flipped = coords_rot.copy()
    center_x = img.shape[1] / 2  # width / 2
    coords_flipped[:, 0] = 2*center_x - coords_flipped[:, 0]


    if gene in ad.var_names:
        # Extract gene expression s
        x = ad[:, gene].X
        values = x.toarray().flatten() if hasattr(x, "toarray") else x.flatten()

        # Plot overlay
        plt.figure(figsize=(6,6))
        plt.imshow(img)
        sc = plt.scatter(coords_flipped[:,0], coords_flipped[:,1],
                         c=values,
                         cmap="viridis",
                         s=8,
                         alpha=0.8)
        plt.colorbar(sc, label=f"{gene} expression")
        plt.title(f"{sample} — {gene}")
        plt.axis("off")
        plt.show()
    else:
        print(f" Gene {gene} not found in {sample}")



In [10]:
for s in samples:
    ad = adatas[s]
    ad.var_names_make_unique()
    sc.pp.highly_variable_genes(ad, n_top_genes=2000)
    sq.gr.spatial_neighbors(ad)
    sq.gr.spatial_autocorr(ad, mode="moran", genes=ad.var.index[ad.var["highly_variable"]])


/Users/ebouvet/Desktop/Spatial_experimenting/.venv/lib/python3.12/site-packages/scipy/sparse/_data.py:144: RuntimeWarning: overflow encountered in expm1
  result = op(self._deduped_data())
/Users/ebouvet/Desktop/Spatial_experimenting/.venv/lib/python3.12/site-packages/scanpy/preprocessing/_highly_variable_genes.py:328: RuntimeWarning: overflow encountered in expm1
  x = np.expm1(x)


ValueError: cannot specify integer `bins` when input data contains infinity

In [9]:
import scanpy as sc
print(sc.__version__)
print(dir(sc)[:20])


1.11.5
['AnnData', 'Neighbors', 'TYPE_CHECKING', 'Verbosity', 'Version', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_compat', '_settings', '_singleton', '_utils']


/var/folders/mz/vr5171gs3kbfntrd2n4mhf640000gp/T/ipykernel_80932/1728815974.py:2: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(sc.__version__)
